In [1]:
!pip install nilearn openneuro-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 96.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.6.1 which is incompatible.
bigframes 1.36.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [23]:
!openneuro-py download --dataset ds003643 --include "stimuli/task-lppCN_section_*" --target-dir "/kaggle/working/ds003643-download/stimuli"  --verify-hash


👋 Hello! This is openneuro-py 2024.2.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds003643 …
📁 Traversing directories for ds003643 : 6302 entities [01:37, 64.65 entities/s] 
📥 Retrieving up to 14 files (5 concurrent downloads). 
CHANGES:   0%|                                        | 0.00/629 [00:00<?, ?B/s]
                                                                                
task-lppCN_section_1.wav:   2%|▏            | 816k/47.5M [00:00<00:05, 8.34MB/s]
task-lppCN_section_5.wav:   0%|                     | 0.00/49.2M [00:00<?, ?B/s]

task-lppCN_section_3.wav:   0%|                     | 0.00/54.1M [00:00<?, ?B/s]


task-lppCN_section_1.wav:   8%|▉           | 3.63M/47.5M [00:00<00:02, 20.9MB/s]



task-lppCN_section_4.wav:   0%|                     | 0.00/51.5M [00:00<?, ?B/s]
task-lppCN_section_5.wav:   1%|             | 390k/49.2M [00:00<00:13, 3.87MB/s]

task-lpp

In [53]:
import os
from tqdm import tqdm
import torchaudio
from transformers import WhisperFeatureExtractor, WhisperModel
import numpy as np
import shutil
import torch.nn as nn
import torch


import math
import nibabel as nib
import nilearn
from nilearn import plotting, image
import matplotlib.pyplot as plt
import pandas as pd

In [11]:
# Set constants
STIMULI_DIR = "/kaggle/working/ds003643-download/stimuli/stimuli/"
SECTIONS = [f"task-lppCN_section_{i}.wav" for i in range(1, 10)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Load Whisper model and feature extractor once
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
model = WhisperModel.from_pretrained("openai/whisper-medium").to(device)

In [51]:
def extract_whisper_features_chunked(audio_path, chunk_seconds=30, window_seconds=2):
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000
    waveform = waveform.squeeze(0)
    total_samples = waveform.shape[0]
    total_duration = total_samples / sample_rate
    chunk_samples = chunk_seconds * sample_rate

    all_frame_features = []

    for i in tqdm(range(0, total_samples, chunk_samples), desc="Processing chunks"):
        chunk = waveform[i:min(i + chunk_samples, total_samples)]
        chunk_np = chunk.numpy()
        inputs = feature_extractor(
            chunk_np,
            sampling_rate=sample_rate,
            return_tensors="pt"
        )
        input_features = inputs.input_features.to(device)
        outputs = model(
            input_features,
            decoder_input_ids=torch.tensor([[50258]], device=device),
            output_hidden_states=True
        )
        chunk_features = outputs.encoder_last_hidden_state.squeeze(0).detach().cpu().numpy()
        all_frame_features.append(chunk_features)

    frame_rate = 50
    expected_frames = int(total_duration * frame_rate)
    frame_features = np.concatenate(all_frame_features, axis=0)[:expected_frames]
    total_frames = frame_features.shape[0]

    frames_per_window = int(window_seconds * frame_rate)
    n_windows = int(np.ceil(total_frames / frames_per_window))

    window_features = []
    for i in range(n_windows):
        start = i * frames_per_window
        end = start + frames_per_window
        window = frame_features[start:end]
        if len(window) > 0:
            window_features.append(window.mean(axis=0))
        else:
            window_features.append(np.zeros(1024))



    # lstm_model = FrameToWindowLSTM()
    # window_features = compute_window_features_with_lstm(frame_features, frames_per_window, lstm_model, device)


    return {
        'window_features': np.array(window_features),
        'window_times': np.arange(n_windows) * window_seconds,
        'frame_features': frame_features,
        'sample_rate': sample_rate
    }

In [54]:
# Loop through and process each file
for filename in SECTIONS:
    audio_path = os.path.join(STIMULI_DIR, filename)
    print(f"\nProcessing {filename}")
    features = extract_whisper_features_chunked(audio_path)
    
    print(f"Shape of window features: {features['window_features'].shape}")
    
    save_path = os.path.join(STIMULI_DIR, filename.replace(".wav", "_features.npy"))
    #np.save(save_path, features)
    
    print(f"Saved features to {save_path}")
    print(features)
    
    # Delete the .wav file
    os.remove(audio_path)
    print(f"Deleted {audio_path}")


Processing task-lppCN_section_1.wav


Processing chunks: 100%|██████████| 19/19 [00:07<00:00,  2.43it/s]


Shape of window features: (283, 256)
Saved features to /kaggle/input/lpp-cnn/task-lppCN_section_1_features.npy
{'window_features': array([[ 0.15240681,  0.10760708, -0.06438774, ..., -0.03494817,
         0.36450106,  0.22801529],
       [-0.12447643,  0.0796905 , -0.14463356, ...,  0.25732663,
         0.00994054,  0.01192469],
       [ 0.13463938,  0.03727895,  0.00722975, ...,  0.09460445,
        -0.00542471,  0.00144633],
       ...,
       [-0.03173942, -0.13551632,  0.09411543, ..., -0.2648737 ,
        -0.01261927,  0.17257902],
       [ 0.11895163, -0.0451399 , -0.08570068, ...,  0.02734895,
         0.19106367, -0.11825727],
       [-0.20882162, -0.22197461, -0.09568503, ...,  0.15109876,
         0.35530037, -0.07123335]], dtype=float32), 'window_times': array([  0,   2,   4,   6,   8,  10,  12,  14,  16,  18,  20,  22,  24,
        26,  28,  30,  32,  34,  36,  38,  40,  42,  44,  46,  48,  50,
        52,  54,  56,  58,  60,  62,  64,  66,  68,  70,  72,  74,  76,
        